# SANTA++ notebook — Windows-local CUDA MiniBatchKMeans copy

This version keeps the original scikit-learn `MiniBatchKMeans` algorithmic choices (greedy k-means++ initialization, `batch_size=4096`, `n_init=1`, online mini-batch updates, low-count reassignment, and early stopping) but executes the expensive distance assignments and centroid updates with PyTorch on CUDA.

It is intended to be much closer to the original notebook than replacing `MiniBatchKMeans` with ordinary full-batch Lloyd k-means. Exact bitwise equality is not guaranteed because CPU and GPU reductions can round differently.

> **First run:** execute the setup/model/fidelity cells only. Before running an experiment cell, set `N_STAT = 1` or `N_INST = 1`; the saved defaults launch multi-instance 8k runs.


In [11]:
# ---- Box Q0: local Windows setup, imports, book ----
import os
import math
import urllib.request
from pathlib import Path

# This model is public. HF_TOKEN is optional, but defining TOKEN keeps the
# rest of the notebook compatible with either authenticated or anonymous use.
TOKEN = os.environ.get("HF_TOKEN")
os.environ["HF_HUB_DISABLE_XET"] = "1"

import numpy as np
import torch
import matplotlib.pyplot as plt
from sklearn.cluster import MiniBatchKMeans

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)
if device == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
print("HF token:", "found" if TOKEN else "not required for this public model")

# Cache the Project Gutenberg text beside the notebook after the first download.
BOOK = Path.cwd() / "monte_cristo.txt"
if BOOK.exists():
    raw = BOOK.read_text(encoding="utf-8")
else:
    url = "https://www.gutenberg.org/cache/epub/1184/pg1184.txt"
    raw = urllib.request.urlopen(url).read().decode("utf-8")
    BOOK.write_text(raw, encoding="utf-8")

off = raw.find("Chapter 1.", raw.find("Chapter 1.") + 10)   # 2nd: body text
text = raw[off:off + 20000]
print("prose window starts:", repr(text[:60]))


device: cuda
GPU: NVIDIA GeForce RTX 5070 Ti Laptop GPU
HF token: not required for this public model
prose window starts: 'Chapter 1. Marseilles—The Arrival\n\n\n\nOn the 24th of February'


In [12]:
# ---- GPU MiniBatchKMeans with scikit-learn-like semantics ----
# This is intentionally not ordinary Lloyd KMeans. It mirrors the algorithm
# used by:
#   MiniBatchKMeans(n_clusters=..., batch_size=4096, n_init=1,
#                   random_state=SEED)
#
# Fidelity choices:
#   * NumPy RandomState controls all stochastic choices, as in scikit-learn.
#   * greedy k-means++ (2 + floor(log(K)) local trials), not vanilla k-means++.
#   * sampling with replacement for each mini-batch.
#   * cumulative-count online means.
#   * scikit-learn's low-count center reassignment and EWA early stopping.
#   * float32 data/centers throughout.
#
# Exact bitwise identity is not guaranteed because GPU and CPU reductions can
# round differently, but on ordinary float32 tests this follows the same path
# extremely closely and can produce identical partitions.

class SklearnLikeTorchMiniBatchKMeans:
    def __init__(
        self,
        n_clusters,
        *,
        batch_size=4096,
        n_init=1,
        max_iter=100,
        tol=0.0,
        max_no_improvement=10,
        init_size=None,
        reassignment_ratio=0.01,
        random_state=0,
        verbose=False,
    ):
        self.n_clusters = int(n_clusters)
        self.batch_size = int(batch_size)
        self.n_init = int(n_init)
        self.max_iter = int(max_iter)
        self.tol = float(tol)
        self.max_no_improvement = max_no_improvement
        self.init_size = init_size
        self.reassignment_ratio = float(reassignment_ratio)
        self.random_state = random_state
        self.verbose = bool(verbose)

    @staticmethod
    def _squared_distances(x, centers):
        """Squared Euclidean distances, [N,D] x [K,D] -> [N,K]."""
        out = (
            x.square().sum(dim=1, keepdim=True)
            + centers.square().sum(dim=1).unsqueeze(0)
        )
        out.addmm_(x, centers.T, beta=1.0, alpha=-2.0)
        return out.clamp_min_(0.0)

    def _assign(self, x, centers):
        dist2 = self._squared_distances(x, centers)
        min_dist2, labels = dist2.min(dim=1)
        return labels, min_dist2.sum()

    def _greedy_kmeans_plus_plus(self, x, rng):
        """GPU version of scikit-learn's greedy k-means++ initializer."""
        n_samples, n_features = x.shape
        n_local_trials = 2 + int(np.log(self.n_clusters))

        # scikit-learn receives float32 unit sample weights for float32 X.
        p = np.ones(n_samples, dtype=np.float32)
        p /= p.sum(dtype=np.float32)
        first_id = int(rng.choice(n_samples, p=p))

        centers = torch.empty(
            self.n_clusters,
            n_features,
            dtype=x.dtype,
            device=x.device,
        )
        centers[0] = x[first_id]

        closest_dist2 = self._squared_distances(
            x, centers[:1]
        ).squeeze(1)
        current_potential = closest_dist2.sum()

        for center_idx in range(1, self.n_clusters):
            # Generate the same kind and number of NumPy random variates as
            # scikit-learn, while doing search/distances on the GPU.
            uniforms = torch.as_tensor(
                rng.uniform(size=n_local_trials),
                dtype=torch.float64,
                device=x.device,
            )
            candidate_values = (
                uniforms * current_potential.to(torch.float64)
            )
            cumulative = torch.cumsum(closest_dist2, dim=0)
            candidate_ids = torch.searchsorted(
                cumulative, candidate_values
            ).clamp_max_(n_samples - 1)

            candidate_points = x[candidate_ids]
            candidate_dist2 = self._squared_distances(
                candidate_points, x
            )
            candidate_dist2 = torch.minimum(
                candidate_dist2, closest_dist2.unsqueeze(0)
            )
            candidate_potentials = candidate_dist2.sum(dim=1)
            best = int(candidate_potentials.argmin().item())

            centers[center_idx] = candidate_points[best]
            closest_dist2 = candidate_dist2[best]
            current_potential = candidate_potentials[best]

        return centers

    @torch.inference_mode()
    def fit_predict(self, x):
        if x.ndim != 2:
            raise ValueError(f"Expected [samples, features], got {tuple(x.shape)}")
        if x.dtype not in (torch.float32, torch.float64):
            x = x.float()
        if not x.is_contiguous():
            x = x.contiguous()

        n_samples = x.shape[0]
        if n_samples < self.n_clusters:
            raise ValueError(
                f"n_samples={n_samples} must be >= n_clusters={self.n_clusters}"
            )

        batch_size = min(self.batch_size, n_samples)
        init_size = self.init_size
        if init_size is None:
            init_size = 3 * batch_size
            if init_size < self.n_clusters:
                init_size = 3 * self.n_clusters
        elif init_size < self.n_clusters:
            init_size = 3 * self.n_clusters
        init_size = min(int(init_size), n_samples)

        rng = np.random.RandomState(self.random_state)

        # scikit-learn draws this validation sample before initialization.
        validation_ids_np = rng.randint(0, n_samples, init_size)
        validation_ids = torch.as_tensor(
            validation_ids_np, dtype=torch.long, device=x.device
        )
        x_valid = x[validation_ids]

        best_centers = None
        best_validation_inertia = None

        for init_idx in range(self.n_init):
            if init_size < n_samples:
                init_ids_np = rng.randint(0, n_samples, init_size)
                init_ids = torch.as_tensor(
                    init_ids_np, dtype=torch.long, device=x.device
                )
                x_init = x[init_ids]
            else:
                x_init = x

            centers = self._greedy_kmeans_plus_plus(x_init, rng)

            # With n_init=1, validation cannot change the selected centers.
            # Skip its expensive assignment while preserving identical output.
            if self.n_init == 1:
                best_centers = centers
            else:
                _, validation_inertia = self._assign(x_valid, centers)
                validation_inertia = float(validation_inertia.item())
                if (
                    best_validation_inertia is None
                    or validation_inertia < best_validation_inertia
                ):
                    best_validation_inertia = validation_inertia
                    best_centers = centers.clone()

        centers = best_centers
        counts = torch.zeros(
            self.n_clusters, dtype=x.dtype, device=x.device
        )

        # scikit-learn scales tol by the mean per-feature variance.
        scaled_tol = (
            float(x.var(dim=0, unbiased=False).mean().item()) * self.tol
            if self.tol > 0.0
            else 0.0
        )

        n_steps = (self.max_iter * n_samples) // batch_size
        ewa_inertia = None
        ewa_inertia_min = None
        no_improvement = 0
        n_since_last_reassign = 0

        for step in range(n_steps):
            n_since_last_reassign += batch_size
            random_reassign = bool(
                (counts == 0).any().item()
                or n_since_last_reassign >= 10 * self.n_clusters
            )
            if random_reassign:
                n_since_last_reassign = 0

            batch_ids_np = rng.randint(0, n_samples, batch_size)
            batch_ids = torch.as_tensor(
                batch_ids_np, dtype=torch.long, device=x.device
            )
            x_batch = x[batch_ids]

            labels, batch_inertia = self._assign(x_batch, centers)

            batch_counts = torch.bincount(
                labels, minlength=self.n_clusters
            ).to(x.dtype)
            batch_sums = torch.zeros_like(centers)
            batch_sums.index_add_(0, labels, x_batch)

            new_counts = counts + batch_counts
            centers_new = centers.clone()
            active = batch_counts > 0
            centers_new[active] = (
                centers[active] * counts[active, None]
                + batch_sums[active]
            ) / new_counts[active, None]
            counts = new_counts

            if random_reassign and self.reassignment_ratio > 0.0:
                to_reassign = (
                    counts
                    < self.reassignment_ratio * counts.max()
                )

                # Same cap as scikit-learn: at most half a mini-batch.
                if int(to_reassign.sum().item()) > 0.5 * x_batch.shape[0]:
                    keep_ids = torch.argsort(counts)[
                        int(0.5 * x_batch.shape[0]):
                    ]
                    to_reassign[keep_ids] = False

                n_reassign = int(to_reassign.sum().item())
                if n_reassign:
                    new_center_rows_np = rng.choice(
                        x_batch.shape[0],
                        replace=False,
                        size=n_reassign,
                    )
                    new_center_rows = torch.as_tensor(
                        new_center_rows_np,
                        dtype=torch.long,
                        device=x.device,
                    )
                    centers_new[to_reassign] = x_batch[new_center_rows]
                    counts[to_reassign] = counts[~to_reassign].min()

            if scaled_tol > 0.0:
                centers_squared_diff = float(
                    (centers_new - centers).square().sum().item()
                )
            else:
                centers_squared_diff = 0.0

            centers = centers_new

            # scikit-learn ignores convergence on the first update.
            if step == 0:
                continue

            mean_batch_inertia = (
                float(batch_inertia.item()) / batch_size
            )
            if ewa_inertia is None:
                ewa_inertia = mean_batch_inertia
            else:
                alpha = min(
                    batch_size * 2.0 / (n_samples + 1), 1.0
                )
                ewa_inertia = (
                    ewa_inertia * (1.0 - alpha)
                    + mean_batch_inertia * alpha
                )

            if (
                scaled_tol > 0.0
                and centers_squared_diff <= scaled_tol
            ):
                break

            if (
                ewa_inertia_min is None
                or ewa_inertia < ewa_inertia_min
            ):
                ewa_inertia_min = ewa_inertia
                no_improvement = 0
            else:
                no_improvement += 1

            if (
                self.max_no_improvement is not None
                and no_improvement >= self.max_no_improvement
            ):
                break

        self.cluster_centers_ = centers
        self.counts_ = counts
        self.n_steps_ = step + 1
        self.n_iter_ = int(
            np.ceil(self.n_steps_ * batch_size / n_samples)
        )

        labels, inertia = self._assign(x, centers)
        self.labels_ = labels
        self.inertia_ = float(inertia.item())

        if self.verbose:
            print(
                f"GPU MiniBatchKMeans: {self.n_steps_} mini-batches, "
                f"{self.n_iter_} effective epochs"
            )

        return labels


def gpu_minibatch_labels(x, n_clusters, seed):
    """Exact parameter match to the original notebook's sklearn call."""
    return SklearnLikeTorchMiniBatchKMeans(
        n_clusters=n_clusters,
        batch_size=4096,
        n_init=1,
        max_iter=100,
        tol=0.0,
        max_no_improvement=10,
        init_size=None,
        reassignment_ratio=0.01,
        random_state=seed,
    ).fit_predict(x)


In [13]:
# ---- Box Q1: load model, one forward pass, cache geometry ----
import time
import transformers
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

t0 = time.time()
print(
    f"torch {torch.__version__} | "
    f"transformers {transformers.__version__} | "
    f"device: {device}"
)

print(f"[{time.time()-t0:5.1f}s] loading tokenizer...")
qtok = AutoTokenizer.from_pretrained(MODEL_NAME)

print(f"[{time.time()-t0:5.1f}s] loading model...")
qmodel = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=torch.float16 if device == "cuda" else torch.float32,
).to(device).eval()

print(f"[{time.time()-t0:5.1f}s] tokenizing and running one forward pass...")
qids = qtok(
    text,
    return_tensors="pt",
    truncation=True,
    max_length=1024,
).input_ids.to(device)

with torch.no_grad():
    qout = qmodel(qids, use_cache=True)

pkv = qout.past_key_values
print(f"[{time.time()-t0:5.1f}s] forward done")


def get_kv(pkv, layer):
    """Access cached K and V for one layer: [B, n_kv, T, d_head]."""
    if hasattr(pkv, "key_cache"):
        return pkv.key_cache[layer], pkv.value_cache[layer]

    if hasattr(pkv, "layers"):
        lyr = pkv.layers[layer]
        return lyr.keys, lyr.values

    return pkv[layer][0], pkv[layer][1]


k0, v0 = get_kv(pkv, 0)
cfg = qmodel.config

print(
    "layers:", cfg.num_hidden_layers,
    "| query heads:", cfg.num_attention_heads,
    "| KV heads:", cfg.num_key_value_heads,
    "| head dim:", k0.shape[-1],
)
print("cached K shape (layer 0):", tuple(k0.shape))
print(
    "tokens:", qids.shape[1],
    "| query heads per KV head:",
    cfg.num_attention_heads // cfg.num_key_value_heads,
)

c:\CondaEnvs\opus-ai\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


torch 2.13.0+cu130 | transformers 5.14.1 | device: cuda
[  0.0s] loading tokenizer...


c:\CondaEnvs\opus-ai\lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\theep\.cache\huggingface\hub\models--Qwen--Qwen2.5-3B-Instruct. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


[  1.9s] loading model...


Loading weights: 100%|██████████| 434/434 [00:01<00:00, 244.13it/s]


[ 83.4s] tokenizing and running one forward pass...
[ 83.9s] forward done
layers: 36 | query heads: 16 | KV heads: 2 | head dim: 128
cached K shape (layer 0): (1, 2, 1024, 128)
tokens: 1024 | query heads per KV head: 8


In [14]:
# ---- Box Q2: fast, restart-safe cache-side fidelity check ----
import math
import time
import torch

L = 512
qids_check = qids[:, :L].contiguous()

nL  = cfg.num_hidden_layers
n_q = cfg.num_attention_heads
n_kv = cfg.num_key_value_heads
gpk = n_q // n_kv
d = k0.shape[-1]

ly, h_q = 0, 0
h_kv = h_q // gpk
T = qids_check.shape[1]

# Q2 must inspect the original Hugging Face attention, not the Q3 patch.
if "forward" in qmodel.model.layers[0].self_attn.__dict__:
    raise RuntimeError(
        "The custom Q3 attention patch is active. "
        "Restart the kernel and rerun Q0, Q1, then Q2."
    )

# An interrupted prior Q2 can leave forward hooks installed.
stale_q_hooks = sum(
    len(layer.self_attn.q_proj._forward_hooks)
    for layer in qmodel.model.layers
)
stale_o_hooks = len(
    qmodel.model.layers[ly].self_attn.o_proj._forward_hooks
)

if stale_q_hooks or stale_o_hooks:
    raise RuntimeError(
        f"Found stale hooks: q_proj={stale_q_hooks}, "
        f"o_proj={stale_o_hooks}. Restart the kernel."
    )

param = next(qmodel.parameters())

print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
print("model device/dtype:", param.device, param.dtype)
print("input device/shape:", qids_check.device, tuple(qids_check.shape))
print(
    "attention implementation:",
    getattr(qmodel.config, "_attn_implementation", "unknown"),
)

assert param.device.type == "cuda", "Model is not on the NVIDIA GPU."
assert qids_check.device.type == "cuda", "Input IDs are not on the NVIDIA GPU."
assert param.dtype == torch.float16, f"Unexpected model dtype: {param.dtype}"

q_raw = {}
attn_out = {}

q_hook = qmodel.model.layers[ly].self_attn.q_proj.register_forward_hook(
    lambda module, inputs, output:
    q_raw.__setitem__(ly, output.detach())
)

o_hook = qmodel.model.layers[ly].self_attn.o_proj.register_forward_hook(
    lambda module, inputs, output:
    attn_out.__setitem__("in", inputs[0].detach())
)

try:
    print("Synchronizing any work left from the previous cell...", flush=True)
    torch.cuda.synchronize()
    print("Previous CUDA work completed.", flush=True)

    torch.cuda.reset_peak_memory_stats()

    print(f"Running bare-model {T}-token fidelity forward...", flush=True)
    t0 = time.perf_counter()

    with torch.inference_mode():
        # Bare Qwen model: returns KV cache but does not compute vocab logits.
        qbase = qmodel.model(
            input_ids=qids_check,
            use_cache=True,
            return_dict=True,
        )

    torch.cuda.synchronize()
    elapsed = time.perf_counter() - t0

    print(f"Forward pass finished in {elapsed:.2f} seconds", flush=True)
    print(
        "Peak allocated VRAM during pass:",
        f"{torch.cuda.max_memory_allocated() / 2**30:.2f} GiB",
    )

finally:
    # This executes even if a normal Python exception occurs.
    q_hook.remove()
    o_hook.remove()

pkv_check = qbase.past_key_values

with torch.inference_mode():
    pos = torch.arange(T, device=device)[None, :]

    dummy = torch.empty(
        1,
        n_q,
        T,
        d,
        device=device,
        dtype=param.dtype,
    )

    cos, sin = qmodel.model.rotary_emb(dummy, pos)


def rotate_half(x):
    x1 = x[..., :d // 2]
    x2 = x[..., d // 2:]
    return torch.cat([-x2, x1], dim=-1)


# Raw q_proj output: [batch, tokens, query_heads * head_dim].
q = q_raw[ly].view(1, T, n_q, d).transpose(1, 2)

# Apply the same RoPE convention used for cached keys.
q = (
    q * cos[:, None, :, :]
    + rotate_half(q) * sin[:, None, :, :]
)

Qr = q[0].float()

Kc, Vc = get_kv(pkv_check, ly)
Kc = Kc[0].float()
Vc = Vc[0].float()

t_last = T - 1

scores = (
    Kc[h_kv, :t_last + 1]
    @ Qr[h_q, t_last]
    / math.sqrt(d)
)

y_ours = (
    torch.softmax(scores, dim=0)
    @ Vc[h_kv, :t_last + 1]
)

y_ref = attn_out["in"][
    0,
    t_last,
    h_q * d:(h_q + 1) * d,
].float()

gate = float(
    (y_ours - y_ref).norm()
    / (y_ref.norm() + 1e-9)
)

print(
    f"cache-side reconstruction, layer {ly} head {h_q}: "
    f"rel diff {gate:.2e}",
    "PASS" if gate < 1e-3
    else "FAIL <- stop here and inspect versions/RoPE",
)

# Q1/Q2 outputs are no longer needed before the 8k experiment.
globals().pop("qout", None)
del qbase, pkv_check, q_raw, attn_out
del Kc, Vc, Qr, q, y_ours, y_ref, scores

torch.cuda.empty_cache()

CUDA available: True
GPU: NVIDIA GeForce RTX 5070 Ti Laptop GPU
model device/dtype: cuda:0 torch.float16
input device/shape: cuda:0 (1, 512)
attention implementation: sdpa
Synchronizing any work left from the previous cell...
Previous CUDA work completed.
Running bare-model 512-token fidelity forward...
Forward pass finished in 0.12 seconds
Peak allocated VRAM during pass: 6.29 GiB
cache-side reconstruction, layer 0 head 0: rel diff 2.03e-04 PASS


In [15]:
# ---- Box Q3: KV2 cache, estimator, stock-matched SDPA patch ----
import math
import types
import torch
from transformers.integrations.sdpa_attention import (
    sdpa_attention_forward as hf_sdpa_attention_forward,
)

# Safe when rerunning Q3 after the old patch was already active.
for lyr in qmodel.model.layers:
    lyr.self_attn.__dict__.pop("forward", None)

CTRL = {"mode": "dense", "S": 64}
KV2 = {}            # layer -> [K, V] as [n_kv, T, d]; our own cache
ess_log = []
read_log = []       # exact token reads per call, topk mode only
qsummaries = {}     # (layer, kv_head) -> dict(group_idx, n_g, kbar_g)


def q_approx_attention(q, K_full, V_full, layer, kv_head, mode, S=64):
    n_total = K_full.shape[0]
    if mode == "dense":
        s = K_full @ q / math.sqrt(d)
        return torch.softmax(s, 0) @ V_full

    s_rec = (K_full[L:] @ q / math.sqrt(d)) if n_total > L else \
            torch.empty(0, device=q.device)

    if mode == "topk":
        sm = qsummaries[(layer, kv_head)]
        gi, n_g, kbar_g = sm["group_idx"], sm["n_g"], sm["kbar_g"]
        guess = torch.log(n_g) + kbar_g @ q / math.sqrt(d)
        order = torch.argsort(guess, descending=True)
        idx, tot = [], 0
        for g in order.tolist():
            idx.append(gi[g])
            tot += len(gi[g])
            if tot >= S:
                break
        j = torch.cat(idx)
        read_log.append(len(j))
        s_all = torch.cat([K_full[j] @ q / math.sqrt(d), s_rec])
        w = torch.softmax(s_all, 0)
        return w @ torch.cat([V_full[j], V_full[L:]])

    if mode == "guided":
        sm = qsummaries[(layer, kv_head)]
        gi, n_g, kbar_g = sm["group_idx"], sm["n_g"], sm["kbar_g"]
        p_group = torch.softmax(torch.log(n_g) + kbar_g @ q / math.sqrt(d), 0)
        g = torch.multinomial(p_group, S, replacement=True)
        j = torch.stack([
            gi[gg][torch.randint(len(gi[gg]), (1,)).item()]
            for gg in g.tolist()
        ])
        log_r = torch.log(p_group[g]) - torch.log(n_g[g])
    elif mode == "santa":
        s_all = K_full[:L] @ q / math.sqrt(d)
        log_p = s_all - torch.logsumexp(s_all, 0)
        j = torch.multinomial(torch.exp(log_p), S, replacement=True)
        log_r = log_p[j]
    elif mode == "uniform":
        j = torch.randint(L, (S,), device=q.device)
        log_r = torch.full((S,), -math.log(L), device=q.device)
    else:
        raise ValueError(f"unknown mode: {mode}")

    s_smp = K_full[j] @ q / math.sqrt(d)
    log_w_smp = s_smp - log_r
    mx = torch.cat([log_w_smp, s_rec]).max() if n_total > L \
         else log_w_smp.max()
    w_smp = torch.exp(log_w_smp - mx) / S
    w_rec = torch.exp(s_rec - mx)
    wn = torch.exp(log_w_smp - log_w_smp.max())
    ess_log.append((layer, kv_head,
                    float(wn.sum()**2 / (wn**2).sum()) / S))
    N = w_smp @ V_full[j] + \
        (w_rec @ V_full[L:] if n_total > L else 0)
    Z = w_smp.sum() + w_rec.sum()
    return N / Z


def q_patched_forward(self, hidden_states, position_embeddings=None,
                      attention_mask=None, *args, **kwargs):
    # This notebook's custom cache is batch-1 only.
    B, T, _ = hidden_states.shape
    if B != 1:
        raise NotImplementedError("Q3's custom KV2 cache supports batch size 1.")
    if position_embeddings is None:
        raise RuntimeError("Qwen did not supply position_embeddings.")

    q = self.q_proj(hidden_states).view(B, T, n_q, d).transpose(1, 2)
    k = self.k_proj(hidden_states).view(B, T, n_kv, d).transpose(1, 2)
    v = self.v_proj(hidden_states).view(B, T, n_kv, d).transpose(1, 2)

    cs, sn = position_embeddings
    def rot(x):
        return x * cs[:, None, :, :] + rotate_half(x) * sn[:, None, :, :]
    q, k = rot(q), rot(k)

    lid = self.layer_id
    if lid in KV2:
        KV2[lid][0] = torch.cat([KV2[lid][0], k[0]], dim=1)
        KV2[lid][1] = torch.cat([KV2[lid][1], v[0]], dim=1)
    else:
        # DynamicCache's first update performs torch.cat(empty, states), which
        # yields contiguous K/V. Match that layout as well as the values.
        KV2[lid] = [k[0].contiguous(), v[0].contiguous()]
    Kf, Vf = KV2[lid]
    Tk = Kf.shape[1]

    if T > 1 or CTRL["mode"] == "dense":
        # Use Transformers 5.12.1's own SDPA wrapper. Unlike the old code,
        # this leaves K/V at n_kv heads and enables the same native GQA path
        # used by stock Qwen whenever no explicit mask is needed.
        if T == Tk:                    # ordinary prefill
            dense_mask = None
            dense_is_causal = T > 1
        elif T == 1:                   # one-token incremental decode
            dense_mask = None
            dense_is_causal = False
        else:                          # multi-token chunk appended to cache
            q_abs = torch.arange(Tk - T, Tk, device=q.device)[:, None]
            k_abs = torch.arange(Tk, device=q.device)[None, :]
            dense_mask = (k_abs <= q_abs)[None, None, :, :]
            dense_is_causal = False

        out, _ = hf_sdpa_attention_forward(
            self,
            q,
            Kf.unsqueeze(0),
            Vf.unsqueeze(0),
            dense_mask,
            dropout=0.0,
            scaling=self.scaling,
            is_causal=dense_is_causal,
        )
        # HF returns [B, T, n_q, d].
    else:
        outs = [
            q_approx_attention(
                q[0, h, 0].float(),
                Kf[h // gpk].float(),
                Vf[h // gpk].float(),
                lid,
                h // gpk,
                CTRL["mode"],
                S=CTRL["S"],
            )
            for h in range(n_q)
        ]
        out = torch.stack(outs)[None, None].to(hidden_states.dtype)

    out = out.reshape(B, T, n_q * d).contiguous()
    return self.o_proj(out), None


def q_repatch():
    for i, lyr in enumerate(qmodel.model.layers):
        lyr.self_attn.layer_id = i
        lyr.self_attn.forward = types.MethodType(
            q_patched_forward, lyr.self_attn
        )


def q_unpatch():
    for lyr in qmodel.model.layers:
        lyr.self_attn.__dict__.pop("forward", None)


# Keep the fidelity prompt fixed even if a later experiment changed global L.
L = 512
qprompt = qids[:, :L].contiguous()


def q_greedy_generate(n_new, return_logit_trace=False):
    """Greedy custom-KV2 incremental generation under the active patch."""
    KV2.clear()
    toks, logit_trace = [], []

    with torch.inference_mode():
        out = qmodel(
            qprompt,
            attention_mask=torch.ones_like(qprompt),
            use_cache=False,
            logits_to_keep=1,
        )
        pos = qprompt.shape[1]

        for step in range(n_new):
            logits = out.logits[0, -1]
            if return_logit_trace:
                logit_trace.append(logits.detach().float().cpu())

            nxt = logits.argmax().view(1, 1)
            toks.append(int(nxt.item()))
            if step + 1 == n_new:
                break

            out = qmodel(
                nxt,
                attention_mask=torch.ones_like(nxt),
                position_ids=torch.full(
                    (1, 1), pos, dtype=torch.long, device=qprompt.device
                ),
                use_cache=False,
                logits_to_keep=1,
            )
            pos += 1

    return (toks, logit_trace) if return_logit_trace else toks


CTRL["mode"] = "dense"
q_repatch()
print("Q3 stock-matched dense patch installed; run the next cell.")

Q3 stock-matched dense patch installed; run the next cell.


In [16]:
# ---- Diagnose dense-patch fidelity with like-for-like decoding ----
import torch

CTRL["mode"] = "dense"
N_FIDELITY_TOKENS = 32


def first_mismatch(a, b):
    for i, (x, y) in enumerate(zip(a, b)):
        if x != y:
            return i, x, y
    if len(a) != len(b):
        return min(len(a), len(b)), None, None
    return None


def stock_cached_trace(n_new):
    """Unpatched Hugging Face incremental decode with past_key_values."""
    toks, logit_trace = [], []
    attn = torch.ones_like(qprompt)

    with torch.inference_mode():
        out = qmodel(
            qprompt,
            attention_mask=attn,
            use_cache=True,
            logits_to_keep=1,
        )
        past = out.past_key_values

        for step in range(n_new):
            logits = out.logits[0, -1]
            logit_trace.append(logits.detach().float().cpu())
            nxt = logits.argmax().view(1, 1)
            toks.append(int(nxt.item()))
            if step + 1 == n_new:
                break

            attn = torch.cat([attn, torch.ones_like(nxt)], dim=1)
            out = qmodel(
                nxt,
                attention_mask=attn,
                past_key_values=past,
                use_cache=True,
                logits_to_keep=1,
            )
            past = out.past_key_values

    return toks, logit_trace


def stock_full_recompute_trace(n_new):
    """Unpatched reference that recomputes the full sequence each step."""
    toks, logit_trace = [], []
    seq = qprompt.clone()

    with torch.inference_mode():
        for _ in range(n_new):
            out = qmodel(
                seq,
                attention_mask=torch.ones_like(seq),
                use_cache=False,
                logits_to_keep=1,
            )
            logits = out.logits[0, -1]
            logit_trace.append(logits.detach().float().cpu())
            nxt = logits.argmax().view(1, 1)
            toks.append(int(nxt.item()))
            seq = torch.cat([seq, nxt], dim=1)

    return toks, logit_trace


def report_pair(name_a, a, name_b, b):
    mismatch = first_mismatch(a, b)
    print(f"\n{name_a} == {name_b}: {a == b}")
    print(f"{name_a}: {qtok.decode(a)!r}")
    print(f"{name_b}: {qtok.decode(b)!r}")

    if mismatch is None:
        print(f"All {min(len(a), len(b))} generated token IDs match.")
    else:
        i, ta, tb = mismatch
        print(
            f"First mismatch at generated token index {i} "
            f"(the {i + 1}th generated token)"
        )
        if ta is not None:
            print(
                f"  {name_a}: id={ta}, "
                f"token={qtok.convert_ids_to_tokens(ta)!r}"
            )
        if tb is not None:
            print(
                f"  {name_b}: id={tb}, "
                f"token={qtok.convert_ids_to_tokens(tb)!r}"
            )
    return mismatch


def report_logit_step(title, trace_a, name_a, trace_b, name_b, step):
    a, b = trace_a[step], trace_b[step]
    delta = a - b
    a_vals, a_ids = torch.topk(a, 2)
    b_vals, b_ids = torch.topk(b, 2)

    print(f"\n{title} — generated token index {step}")
    print("max absolute logit difference:", float(delta.abs().max()))
    print("RMS logit difference:", float(delta.square().mean().sqrt()))
    print(f"{name_a} top-1/top-2 margin:", float(a_vals[0] - a_vals[1]))
    print(f"{name_b} top-1/top-2 margin:", float(b_vals[0] - b_vals[1]))
    print(
        f"{name_a} top IDs:", a_ids.tolist(),
        [qtok.convert_ids_to_tokens(int(x)) for x in a_ids],
    )
    print(
        f"{name_b} top IDs:", b_ids.tolist(),
        [qtok.convert_ids_to_tokens(int(x)) for x in b_ids],
    )


# Patched custom-KV2 incremental path.
q_repatch()
dense_patched_test, patched_logits = q_greedy_generate(
    N_FIDELITY_TOKENS,
    return_logit_trace=True,
)

# Two unpatched Hugging Face references.
q_unpatch()
try:
    dense_stock_cached, stock_cached_logits = stock_cached_trace(
        N_FIDELITY_TOKENS
    )
    dense_stock_full, stock_full_logits = stock_full_recompute_trace(
        N_FIDELITY_TOKENS
    )
finally:
    # Keep the patch active for the later experiment cells.
    q_repatch()


main_mismatch = report_pair(
    "patched incremental",
    dense_patched_test,
    "stock cached",
    dense_stock_cached,
)
report_pair(
    "stock cached",
    dense_stock_cached,
    "stock full-recompute",
    dense_stock_full,
)
report_pair(
    "patched incremental",
    dense_patched_test,
    "stock full-recompute",
    dense_stock_full,
)

# This verifies whether there is any first-token discrepancy.
report_logit_step(
    "First-step logit comparison",
    patched_logits, "patched incremental",
    stock_cached_logits, "stock cached",
    0,
)

# If text still forks, inspect the logits that chose the first differing token.
if main_mismatch is not None:
    report_logit_step(
        "Logits at the patched-vs-stock fork",
        patched_logits, "patched incremental",
        stock_cached_logits, "stock cached",
        main_mismatch[0],
    )
else:
    per_step_max = torch.tensor([
        (a - b).abs().max().item()
        for a, b in zip(patched_logits, stock_cached_logits)
    ])
    worst = int(per_step_max.argmax())
    print(
        f"\nExact generated-token parity achieved for "
        f"{N_FIDELITY_TOKENS} tokens."
    )
    print(
        "Largest per-step max-absolute logit difference:",
        float(per_step_max[worst]),
        "at generated token index",
        worst,
    )


patched incremental == stock cached: True
patched incremental: ' as the night. He was dressed in a\n\nlight-colored coat, and a light-colored shirt, and wore a pair of\n\nlight-colored trousers, and a'
stock cached: ' as the night. He was dressed in a\n\nlight-colored coat, and a light-colored shirt, and wore a pair of\n\nlight-colored trousers, and a'
All 32 generated token IDs match.

stock cached == stock full-recompute: True
stock cached: ' as the night. He was dressed in a\n\nlight-colored coat, and a light-colored shirt, and wore a pair of\n\nlight-colored trousers, and a'
stock full-recompute: ' as the night. He was dressed in a\n\nlight-colored coat, and a light-colored shirt, and wore a pair of\n\nlight-colored trousers, and a'
All 32 generated token IDs match.

patched incremental == stock full-recompute: True
patched incremental: ' as the night. He was dressed in a\n\nlight-colored coat, and a light-colored shirt, and wore a pair of\n\nlight-colored trousers, and a'
stock f

In [17]:
# ---- Box Q4: pass key at 8k: dense vs SANTA vs guided ----
FILL_LEN  = 8192
N_INST    = 10
B_FILL    = 16       # nominal group size (coarse on purpose)
S_KEY     = 128       # samples per head per decode step
P_PROBE   = 64       # probe queries per sharing head for fingerprints
W_RECENT  = 64       # most recent prefill tokens always read exactly
N_GEN     = 8
SEED      = 0
MODES     = ["dense", "santa", "guided"]

filler_ids = qtok(raw[off:off + 400000], add_special_tokens=False,
                  return_tensors="pt").input_ids[0].to(device)
assert len(filler_ids) >= FILL_LEN + 1000
rng = np.random.RandomState(SEED)

def make_instance():
    key = "".join(rng.choice(list("123456789"), 5))
    needle = qtok(f"\n\nThe pass key is {key}. Remember it. "
                  f"The pass key is {key}.\n\n", add_special_tokens=False,
                  return_tensors="pt").input_ids[0].to(device)
    question = qtok("\n\nWhat is the pass key? The pass key is",
                    add_special_tokens=False,
                    return_tensors="pt").input_ids[0].to(device)
    pos = int(rng.uniform(0.05, 0.85) * FILL_LEN)
    ctx = torch.cat([filler_ids[:pos], needle,
                     filler_ids[pos:FILL_LEN], question])[None]
    return key, pos / FILL_LEN, ctx

def prefill_and_cluster(ctx):
    """Dense prefill once; fingerprint-cluster tokens 0..SAMPLE_END-1.
    Everything from SAMPLE_END on (recent window, refed token, generated
    tokens) is always read exactly, so nothing is counted twice."""
    global L
    L_ctx = ctx.shape[1]
    SAMPLE_END = L_ctx - 1 - W_RECENT
    KV2.clear()
    CTRL["mode"] = "dense"
    q_raw9 = {}
    hooks = [qmodel.model.layers[i].self_attn.q_proj.register_forward_hook(
                (lambda i: lambda m, inp, out:
                 q_raw9.__setitem__(i, out.detach()))(i)) for i in range(nL)]
    with torch.no_grad():
        _ = qmodel(ctx, use_cache=False)
    for h in hooks:
        h.remove()

    with torch.no_grad():
        p = torch.arange(L_ctx, device=device)[None, :]
        dm = torch.zeros(1, L_ctx, n_q, d, device=device).transpose(1, 2)
        cs9, sn9 = qmodel.model.rotary_emb(dm, p)

    probe = np.sort(rng.choice(np.arange(3 * L_ctx // 4, L_ctx - 1),
                               size=P_PROBE, replace=False))
    probe_t = torch.tensor(probe, device=device)

    for ly in range(nL):
        qr = q_raw9[ly].view(1, L_ctx, n_q, d).transpose(1, 2)
        qr = (qr * cs9[:, None, :, :] +
              rotate_half(qr) * sn9[:, None, :, :])[0].float()
        Kf, _ = KV2[ly]
        for hk in range(n_kv):
            Kx = Kf[hk, :SAMPLE_END].float()
            G = torch.cat([Kx @ qr[h, probe_t].T / math.sqrt(d)
                           for h in range(hk * gpk, (hk + 1) * gpk)],
                          dim=1).contiguous()
            # Match NumPy's population-standard-deviation normalization.
            G = ((G - G.mean(dim=0, keepdim=True)) /
                 (G.std(dim=0, keepdim=True, unbiased=False) + 1e-6))
            n_cl = max(2, SAMPLE_END // B_FILL)
            lab_t = gpu_minibatch_labels(G, n_cl, SEED).long()
            gi = [ix for c in range(n_cl)
                  if len(ix := torch.where(lab_t == c)[0]) > 0]
            qsummaries[(ly, hk)] = dict(
                group_idx=gi,
                n_g=torch.tensor([len(ix) for ix in gi],
                                 device=device).float(),
                kbar_g=torch.stack([Kx[ix].mean(0) for ix in gi]))
    del q_raw9
    snap = {k: [v[0][:, :L_ctx - 1].clone(), v[1][:, :L_ctx - 1].clone()]
            for k, v in KV2.items()}
    return snap, SAMPLE_END

def generate_from(snap, ctx, SAMPLE_END, mode):
    global L
    KV2.clear()
    KV2.update({k: [v[0], v[1]] for k, v in snap.items()})
    L = SAMPLE_END
    CTRL["mode"], CTRL["S"] = mode, S_KEY
    L_ctx = ctx.shape[1]
    toks, inp, pos = [], ctx[:, -1:], L_ctx - 1
    with torch.no_grad():
        for _ in range(N_GEN):
            out = qmodel(inp, position_ids=torch.tensor([[pos]], device=device),
                         use_cache=False)
            inp = out.logits[0, -1].argmax().view(1, 1)
            pos += 1
            toks.append(int(inp))
    return qtok.decode(toks)

inst = [make_instance() for _ in range(N_INST)]
L_ctx = inst[0][2].shape[1]
print(f"context ~{L_ctx} tokens | B={B_FILL} S={S_KEY} | {N_INST} instances\n")

results, ess_by = {m: [] for m in MODES}, {}
torch.manual_seed(SEED)
for i, (key, depth, ctx) in enumerate(inst):
    snap, SAMPLE_END = prefill_and_cluster(ctx)
    for m in MODES:
        ess_log.clear()
        txt = generate_from(snap, ctx, SAMPLE_END, m)
        results[m].append(key in txt)
        if m in ("guided", "santa", "uniform"):
            ess_by.setdefault(m, []).append(np.mean([e for _, _, e in ess_log]))
        print(f"  inst {i} depth {depth:.2f} key {key} {m:7s} "
              f"{'HIT ' if key in txt else 'miss'} | {txt!r}")

M_g = SAMPLE_END // B_FILL
extra = W_RECENT + 1 + N_GEN / 2
def traffic(m):
    if m == "dense":  return 100.0
    if m == "santa":  return 100 * (0.5 + (S_KEY / 2 + extra) / L_ctx)
    return 100 * (M_g / 2 + S_KEY + extra) / L_ctx

print(f"\n{'mode':7s} {'retrieval':>9s} {'ESS/S':>6s} {'traffic':>8s}")
for m in MODES:
    e = f"{np.mean(ess_by[m]):.2f}" if m in ess_by else ""
    print(f"{m:7s} {np.mean(results[m]):9.2f} {e:>6s} {traffic(m):7.1f}%")


context ~8229 tokens | B=16 S=128 | 10 instances

  inst 0 depth 0.39 key 61448 dense   HIT  | ' 61448. Remember'
  inst 0 depth 0.39 key 61448 santa   HIT  | ' 61448. Remember'
  inst 0 depth 0.39 key 61448 guided  HIT  | ' 61448. The'
  inst 1 depth 0.10 key 63587 dense   HIT  | ' 63587.\n\n Chapter'
  inst 1 depth 0.10 key 63587 santa   HIT  | ' 63587.\n\n Chapter'
  inst 1 depth 0.10 key 63587 guided  miss | ' 8345.\n\n00'


KeyboardInterrupt: 